In [9]:
import pandas as pd
import numpy as np

# Load cleaned datasets
listings = pd.read_csv("listings_cleaned.csv", low_memory=False)
sold = pd.read_csv("sold_cleaned.csv", low_memory=False)

In [10]:
def engineer_features(df, name="dataset"):
    df = df.copy()

    # -----------------------------
    # 1. Convert numeric columns
    # -----------------------------
    numeric_cols = [
        "ClosePrice",
        "OriginalListPrice",
        "LivingArea",
        "DaysOnMarket"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -----------------------------
    # 2. Convert date columns
    # -----------------------------
    date_cols = [
        "CloseDate",
        "PurchaseContractDate",
        "ListingContractDate"
    ]

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # -----------------------------
    # 3. Price Ratio
    # ClosePrice / OriginalListPrice
    # -----------------------------
    if {"ClosePrice", "OriginalListPrice"}.issubset(df.columns):
        df["price_ratio"] = df["ClosePrice"] / df["OriginalListPrice"].replace(0, np.nan)

    # -----------------------------
    # 4. Close to Original List Ratio
    # Same formula, required separately by handbook
    # -----------------------------
    if {"ClosePrice", "OriginalListPrice"}.issubset(df.columns):
        df["close_to_original_list_ratio"] = (
            df["ClosePrice"] / df["OriginalListPrice"].replace(0, np.nan)
        )

    # -----------------------------
    # 5. Price Per Sq Ft
    # ClosePrice / LivingArea
    # -----------------------------
    if {"ClosePrice", "LivingArea"}.issubset(df.columns):
        df["price_per_sqft"] = df["ClosePrice"] / df["LivingArea"].replace(0, np.nan)

    # -----------------------------
    # 6. Days on Market
    # Raw field already exists
    # -----------------------------
    if "DaysOnMarket" in df.columns:
        df["days_on_market"] = df["DaysOnMarket"]

    # -----------------------------
    # 7. Year / Month / YrMo
    # Derived from CloseDate
    # -----------------------------
    if "CloseDate" in df.columns:
        df["year"] = df["CloseDate"].dt.year
        df["month"] = df["CloseDate"].dt.month
        df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

    # -----------------------------
    # 8. Listing to Contract Days
    # PurchaseContractDate - ListingContractDate
    # -----------------------------
    if {"PurchaseContractDate", "ListingContractDate"}.issubset(df.columns):
        df["listing_to_contract_days"] = (
            df["PurchaseContractDate"] - df["ListingContractDate"]
        ).dt.days

    # -----------------------------
    # 9. Contract to Close Days
    # CloseDate - PurchaseContractDate
    # -----------------------------
    if {"CloseDate", "PurchaseContractDate"}.issubset(df.columns):
        df["contract_to_close_days"] = (
            df["CloseDate"] - df["PurchaseContractDate"]
        ).dt.days

    # -----------------------------
    # 10. Data Quality Flags
    # -----------------------------
    if "price_ratio" in df.columns:
        df["extreme_price_ratio_flag"] = df["price_ratio"] > 2

    if "contract_to_close_days" in df.columns:
        df["negative_contract_to_close_flag"] = df["contract_to_close_days"] < 0

    if "listing_to_contract_days" in df.columns:
        df["negative_listing_to_contract_flag"] = df["listing_to_contract_days"] < 0

    # -----------------------------
    # 11. Sample Output Table
    # -----------------------------
    print(f"\n==============================")
    print(f"Sample Engineered Data: {name}")
    print(f"==============================")

    cols_to_show = [
        "ClosePrice",
        "OriginalListPrice",
        "price_ratio",
        "close_to_original_list_ratio",
        "price_per_sqft",
        "DaysOnMarket",
        "days_on_market",
        "year",
        "month",
        "YrMo",
        "listing_to_contract_days",
        "contract_to_close_days"
    ]

    existing_cols = [col for col in cols_to_show if col in df.columns]
    display(df[existing_cols].head(10))

    return df  

In [11]:
# Apply feature engineering

# Sold data is the main dataset for these metrics because it has ClosePrice and CloseDate.
sold_fe = engineer_features(sold, "Sold")

# Listings data may have missing ClosePrice and CloseDate, but we can still run it if needed.
listings_fe = engineer_features(listings, "Listings")


Sample Engineered Data: Sold


,ClosePrice,OriginalListPrice,price_ratio,close_to_original_list_ratio,price_per_sqft,DaysOnMarket,days_on_market,year,month,YrMo,listing_to_contract_days,contract_to_close_days
0,95000.0,98000.0,0.969388,0.969388,69.444444,86.0,86.0,2022,2,2022-02,107.0,28.0
1,1200.0,1200.0,1.000000,1.000000,1.411765,125.0,125.0,2022,2,2022-02,129.0,0.0
2,1100000.0,1100000.0,1.000000,1.000000,818.452381,106.0,106.0,2022,4,2022-04,113.0,71.0
3,2499999.0,2499999.0,1.000000,1.000000,945.179206,37.0,37.0,2022,1,2022-01,37.0,46.0
4,640000.0,598888.0,1.068647,1.068647,534.223706,15.0,15.0,2022,1,2022-01,65.0,26.0
5,640000.0,565000.0,1.132743,1.132743,309.178744,13.0,13.0,2022,1,2022-01,14.0,76.0
6,438000.0,349999.0,1.251432,1.251432,373.083475,3.0,3.0,2022,3,2022-03,3.0,158.0
7,325000.0,325500.0,0.998464,0.998464,401.234568,9.0,9.0,2022,4,2022-04,9.0,174.0
8,248000.0,219250.0,1.131129,1.131129,215.464813,112.0,112.0,2022,2,2022-02,112.0,12.0
9,5900.0,7280.0,0.810440,0.810440,3.113456,141.0,141.0,2022,3,2022-03,141.0,4.0



Sample Engineered Data: Listings


,ClosePrice,OriginalListPrice,price_ratio,close_to_original_list_ratio,price_per_sqft,DaysOnMarket,days_on_market,year,month,YrMo,listing_to_contract_days,contract_to_close_days
0,1300000.0,1300000.0,1.000000,1.000000,669.412976,0,0,2026,4,2026-04,61,0
1,1529000.0,1529000.0,1.000000,1.000000,668.853893,0,0,2026,4,2026-04,16,32
2,795000.0,795000.0,1.000000,1.000000,479.493366,0,0,2026,4,2026-04,64,0
3,339000.0,339000.0,1.000000,1.000000,253.932584,0,0,2026,4,2026-04,65,0
4,5650000.0,5895000.0,0.958439,0.958439,1576.011158,62,62,2026,4,2026-04,62,0
5,1950000.0,1950000.0,1.000000,1.000000,574.374080,0,0,2026,4,2026-04,0,45
6,1390000.0,1390000.0,1.000000,1.000000,1268.248175,0,0,2026,3,2026-03,0,54
7,1300000.0,1300000.0,1.000000,1.000000,421.530480,0,0,2026,4,2026-04,24,22
8,525000.0,525000.0,1.000000,1.000000,330.396476,37,37,2026,4,2026-04,37,14
9,400000.0,349900.0,1.143184,1.143184,173.611111,0,0,2026,4,2026-04,4,53


In [12]:
# Function to create segmented summary tables

def create_segment_summary(df, segment_col, name="dataset"):
    metrics = [
        "ClosePrice",
        "price_ratio",
        "close_to_original_list_ratio",
        "price_per_sqft",
        "DaysOnMarket",
        "listing_to_contract_days",
        "contract_to_close_days"
    ]

    existing_metrics = [col for col in metrics if col in df.columns]

    if segment_col not in df.columns:
        print(f"{segment_col} does not exist in {name}.")
        return None

    if len(existing_metrics) == 0:
        print(f"No engineered metrics available for {segment_col} summary.")
        return None

    summary = (
        df.groupby(segment_col)[existing_metrics]
        .agg(["count", "mean", "median", "min", "max"])
        .round(2)
    )

    print(f"\n==============================")
    print(f"{name} Segment Summary by {segment_col}")
    print(f"==============================")

    display(summary.head(20))

    return summary

In [13]:
# Required segmented summary table
# Handbook says at least one grouped by PropertyType or CountyOrParish.

property_type_summary = create_segment_summary(
    sold_fe,
    "PropertyType",
    "Sold"
)

county_summary = create_segment_summary(
    sold_fe,
    "CountyOrParish",
    "Sold"
)


Sold Segment Summary by PropertyType


ClosePrice                                                 \
                        count        mean      median       min          max   
PropertyType                                                                   
CommercialLease             5     2165.13      673.75       1.9       6500.0   
CommercialSale             18  1366800.00  1150000.00  350000.0    2750000.0   
ManufacturedInPark      18216   222527.41   180000.00       0.0  140000000.0   
Residential            502097  1165339.96   818000.00       0.0  989500000.0   
ResidentialIncome        4980  1866360.20  1509000.00   78500.0   28000000.0   
ResidentialLease       169403     7962.85     3750.00       0.0   28000000.0   

                   price_ratio                                  ...  \
                         count   mean median   min         max  ...   
PropertyType                                                    ...   
CommercialLease              5   0.96   0.97  0.90        1.00  ...   
CommercialSale              18   0.90   0.94  0.71        1.00  ...   
ManufacturedInPark       18205   1.42   0.95  0.00     1007.25  ...   
Residential             501154  51.39   1.00  0.00  9118000.00  ...   
ResidentialIncome         4940   1.12   0.96  0.01      840.58  ...   
ResidentialLease        169386   1.69   1.00  0.00    10000.00  ...   

                   listing_to_contract_days                                   \
                                      count    mean median      min      max   
PropertyType                                                                   
CommercialLease                           5  110.00   35.0     20.0    357.0   
CommercialSale                           18  116.89   91.0      0.0    319.0   
ManufacturedInPark                    18210   67.47   43.0   -199.0   1451.0   
Residential                          501897   42.81   22.0 -36407.0  14657.0   
ResidentialIncome                      4966   59.10   35.0     -1.0   2616.0   
ResidentialLease                     155626   43.33   27.0    -38.0   8213.0   

                   contract_to_close_days                                
                                    count   mean median    min      max  
PropertyType                                                             
CommercialLease                         5   4.60    1.0    0.0     18.0  
CommercialSale                         18  27.33   12.5    0.0    127.0  
ManufacturedInPark                  18211  34.56   31.0   -6.0    593.0  
Residential                        501898  31.98   29.0 -491.0  36629.0  
ResidentialIncome                    4966  40.27   32.0 -447.0   1507.0  
ResidentialLease                   155627   4.63    1.0 -485.0   3652.0  

[6 rows x 35 columns]


Sold Segment Summary by CountyOrParish


ClosePrice                                                 \
                     count        mean     median        min          max   
CountyOrParish                                                              
Alameda              25595  1210980.70  1070000.0      464.0  765000000.0   
Alpine                   1  1100000.00  1100000.0  1100000.0    1100000.0   
Amador                  38   346037.76   317500.0      900.0     855000.0   
Butte                 5398   401919.25   380000.0       25.0    5550000.0   
Calaveras              108   553577.46   480500.0     1650.0    2600000.0   
Clark                    1   375000.00   375000.0   375000.0     375000.0   
Colusa                  29   412048.28   410000.0     9500.0    1100000.0   
Contra Costa         25592  1004184.57   765000.0      345.0  600000000.0   
Del Norte                5  1226300.00   570000.0    45000.0    2982500.0   
El Dorado              128   669647.60   620000.0     1695.0    2195000.0   
Foreign Country          8   529608.25   272500.0    90000.0    1850000.0   
Fresno                1748   465081.80   400000.0     1200.0    5600000.0   
Glenn                  477   345383.77   344900.0      925.0     950000.0   
Humboldt                15   376929.67   420000.0     1595.0     815000.0   
Imperial               375   283531.59   300000.0        0.0     825000.0   
Inyo                    10   631950.00   567250.0   105000.0    2400000.0   
Kern                  4952   536131.57   349925.0      525.0  900000000.0   
Kings                  262   372827.49   345000.0    35000.0    3550000.0   
Lake                  2615   294596.15   284500.0      700.0    3000000.0   
Lassen                  17   245088.24   200000.0    46000.0     695000.0   

                price_ratio                              ...  \
                      count  mean median   min      max  ...   
CountyOrParish                                           ...   
Alameda               25563  1.40   1.02  0.00  1098.04  ...   
Alpine                    1  0.67   0.67  0.67     0.67  ...   
Amador                   37  0.91   0.92  0.51     1.03  ...   
Butte                  5397  1.70   0.98  0.00  1038.96  ...   
Calaveras               106  0.93   0.96  0.40     1.21  ...   
Clark                     1  0.97   0.97  0.97     0.97  ...   
Colusa                   29  0.94   1.00  0.43     1.08  ...   
Contra Costa          25575  1.17   1.00  0.00  1224.23  ...   
Del Norte                 5  0.90   0.92  0.76     0.99  ...   
El Dorado               128  0.95   0.97  0.55     2.10  ...   
Foreign Country           8  0.97   1.00  0.78     1.23  ...   
Fresno                 1748  2.72   0.99  0.10  2066.00  ...   
Glenn                   477  0.95   0.98  0.46     1.56  ...   
Humboldt                 15  0.89   0.92  0.52     1.07  ...   
Imperial                375  0.96   0.99  0.00     1.97  ...   
Inyo                     10  0.90   0.95  0.56     1.00  ...   
Kern                   4951  1.96   0.99  0.01  1000.00  ...   
Kings                   262  4.83   1.00  0.58  1000.00  ...   
Lake                   2615  0.93   0.95  0.27     9.23  ...   
Lassen                   17  0.87   0.90  0.63     1.00  ...   

                listing_to_contract_days                                \
                                   count    mean median    min     max   
CountyOrParish                                                           
Alameda                            25594   26.45   14.0    0.0   801.0   
Alpine                                 1  231.00  231.0  231.0   231.0   
Amador                                38  103.13   59.0    1.0   886.0   
Butte                               5398   58.31   29.0    0.0   812.0   
Calaveras                            107   64.39   46.0    0.0   457.0   
Clark                                  1    2.00    2.0    2.0     2.0   
Colusa                                29   58.34   11.0    0.0   263.0   
Contra Costa                   

In [14]:
# Optional extra segment summaries from the handbook

property_subtype_summary = create_segment_summary(
    sold_fe,
    "PropertySubType",
    "Sold"
)

mls_area_summary = create_segment_summary(
    sold_fe,
    "MLSAreaMajor",
    "Sold"
)

list_office_summary = create_segment_summary(
    sold_fe,
    "ListOfficeName",
    "Sold"
)

buyer_office_summary = create_segment_summary(
    sold_fe,
    "BuyerOfficeName",
    "Sold"
)


Sold Segment Summary by PropertySubType


ClosePrice                                   \
                           count        mean     median       min   
PropertySubType                                                     
Apartment                  14752     5229.87     2500.0       0.0   
BoatSlip                      55   210787.27   185000.0   37000.0   
Cabin                        612   255048.67   212500.0    1200.0   
CoOwnership                   18   780143.28   439999.5  220000.0   
Condominium               129014   550792.93   465000.0       0.0   
DeededParking                  6   438400.00   517500.0     700.0   
Duplex                      9107   481363.44     4100.0       0.0   
Farm                          24  2825145.83  1675000.0  490000.0   
Loft                          92   269898.13     3700.0       0.0   
ManufacturedHome              63   197864.73   194000.0     925.0   
ManufacturedOnLand          7589   317768.46   300000.0     500.0   
MixedUse                     426   776274.26   545000.0     200.0   
MobileHome                    85   290356.34   170000.0    1150.0   
OwnYourOwn                   102   275479.10   260000.0     875.0   
Quadruplex                  2067   410362.90     2750.0     200.0   
RoomingHouse                 343     1567.82     1350.0     550.0   
SingleFamilyResidence     455537  1047660.87   770000.0       0.0   
StockCooperative            2541   309023.82   329000.0    1350.0   
Studio                       540    11570.33     1750.0      10.0   
Timeshare                     30   356083.33   217500.0    3500.0   

                                   price_ratio                       \
                               max       count    mean median   min   
PropertySubType                                                       
Apartment                8122023.0       14749    1.67   1.00  0.00   
BoatSlip                  450000.0          55    0.93   0.95  0.65   
Cabin                    2989000.0         612    0.90   0.92  0.09   
CoOwnership              5250000.0          18   56.54   0.98  0.88   
Condominium            890000000.0      128877   14.30   1.00  0.00   
DeededParking             960000.0           6    0.82   0.97  0.00   
Duplex                  14000000.0        9085  108.42   1.00  0.00   
Farm                     9650000.0          23    0.93   0.95  0.47   
Loft                     1315000.0          92    0.97   1.00  0.00   
ManufacturedHome          597000.0          63    0.96   0.98  0.69   
ManufacturedOnLand       5372223.0        7589    1.46   0.97  0.00   
MixedUse                 8325000.0         424    0.92   0.95  0.01   
MobileHome               1350000.0          85    0.91   0.94  0.10   
OwnYourOwn               1400000.0         102    0.93   0.96  0.61   
Quadruplex               8142023.0        2056    2.66   1.00  0.09   
RoomingHouse               14500.0         343    1.03   1.00  0.33   
SingleFamilyResidence  989500000.0      454796   48.63   1.00  0.00   
StockCooperative         3690000.0        2541    1.45   0.99  0.45   
Studio                    590000.0         540    1.01   1.00  0.01   
Timeshare                1225000.0          30    0.90   0.98  0.45   

                                   ... listing_to_contract_days          \
                              max  ...                    count    mean   
PropertySubType                    ...                                    
Apartment                 3895.45  ...                    13469   47.41   
BoatSlip                     1.12  ...                       55   46.24   
Cabin                        6.59  ...                      612   85.02   
CoOwnership               1000.23  ...                       18   70.89   
Condominium             945736.43  ...                   122952   44.22   
DeededParking                1.06  ...                        6   30.67   
Duplex                  972222.22  ...                     9107   45.40   
Farm                         1.14  ...            


Sold Segment Summary by MLSAreaMajor


ClosePrice              \
                                                       count        mean   
MLSAreaMajor                                                               
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr        845  1055813.66   
101 - North Inglewood                                    895   456029.08   
102 - South Inglewood                                    332   559491.99   
103 - Ladera Heights                                     221  1065057.32   
105 - Lennox                                              52   565063.94   
106 - Los Angeles                                         22   971465.91   
107 - Holly Glen/Del Aire                                373   720374.71   
108 - North Hawthorne                                    200   437393.40   
109 - Ramona/Burleigh                                    130   484820.01   
11 - Westside                                            259   551839.98   
110 - East Hawthorne                                     256   382701.26   
111 - Bodger Park/El Camino                              169   739367.76   
112 - North Lawndale                                     157   532348.90   
113 - South Lawndale                                     122   545211.34   
114 - Hollypark                                           90   626127.03   
115 - North Gardena                                      136   542567.90   
116 - North Gateway                                       98   629214.74   
117 - McCarthy                                           204   572195.32   
118 - Pacific Square                                     167   476618.92   
119 - Central Gardena                                    290   400098.93   

                                                                      \
                                                      median     min   
MLSAreaMajor                                                           
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr   795000.0  1400.0   
101 - North Inglewood                               435000.0  1100.0   
102 - South Inglewood                               709000.0   400.0   
103 - Ladera Heights                               1250000.0  2000.0   
105 - Lennox                                        602500.0  1475.0   
106 - Los Angeles                                   867500.0  3000.0   
107 - Holly Glen/Del Aire                           910000.0  2200.0   
108 - North Hawthorne                               444000.0  1295.0   
109 - Ramona/Burleigh                               278750.0  1250.0   
11 - Westside                                       635000.0  1350.0   
110 - East Hawthorne                                  4597.5  1300.0   
111 - Bodger Park/El Camino                         835000.0  1850.0   
112 - North Lawndale                                627000.0  1995.0   
113 - South Lawndale                                697500.0  1995.0   
114 - Hollypark                                     772000.0  1850.0   
115 - North Gardena                                 711750.0  1600.0   
116 - North Gateway                                 720000.0  1350.0   
117 - McCarthy                                      722500.0  1750.0   
118 - Pacific Square                                545000.0  1150.0   
119 - Central Gardena                               455750.0  1749.0   

                                                              price_ratio  \
                                                          max       count   
MLSAreaMajor                                                                
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr  10500000.0         845   
101 - North Inglewood                              10200000.0         895   
102 - South Inglewood                               4400000.0         332   
103 - Ladera Heights                                3550000.0         221   
105 - Lennox                                        4700000.0          52   
106 - Los Angeles           


Sold Segment Summary by ListOfficeName


ClosePrice                                   \
                                count        mean     median       min   
ListOfficeName                                                           
#1 FLAT FEE-LIBERTY REALTY          2   922500.00   922500.0  900000.0   
#1 Home Sales®                      2   623500.00   623500.0  412000.0   
#1 Home SalesÂ®                     1   545000.00   545000.0  545000.0   
#1 Realty Homes Inc.                6     2350.00     2250.0    1700.0   
*                                   2    46750.00    46750.0    3500.0   
1 Percent Lists SoCal              27   523025.30   536000.0    2795.0   
1 Realty                           29   226708.62   190000.0    1950.0   
1 STOP REALTY LLC                   4     3285.00     3425.0    2295.0   
1 Team Realtors                    13   873876.92   718400.0  355000.0   
1 Vision Real Estate               11   579318.18   455000.0  120000.0   
1% LISTING FEE                     36   606709.72   565000.0    2000.0   
1% Listing Broker                  64   970052.95   955000.0    4500.0   
1-2-3 Homekeys                      1   824000.00   824000.0  824000.0   
10-8 Real Estate                    4  1635000.00  1675000.0  690000.0   
10-Point-o Inc.                     2  1303000.00  1303000.0  706000.0   
100 Doors Realty, Inc.             10   407655.35   570000.0       3.5   
1000 Realty Corporation            43   544286.98   502500.0    1700.0   
101 Realty Services                 3  2020129.33   489000.0   12500.0   
101 Realty, Inc.                    3  1208333.33   770000.0  705000.0   
10X Realty Mortgage                29   852999.97   773000.0  300000.0   

                                      price_ratio                           \
                                  max       count  mean median   min   max   
ListOfficeName                                                               
#1 FLAT FEE-LIBERTY REALTY   945000.0           2  0.96   0.96  0.95  0.98   
#1 Home Sales®               835000.0           2  0.92   0.92  0.84  1.00   
#1 Home SalesÂ®              545000.0           1  0.84   0.84  0.84  0.84   
#1 Realty Homes Inc.           3200.0           6  0.99   1.00  0.91  1.00   
*                             90000.0           2  0.88   0.88  0.75  1.00   
1 Percent Lists SoCal       1825000.0          27  1.01   1.00  0.90  1.13   
1 Realty                    1730000.0          29  0.99   0.96  0.77  1.73   
1 STOP REALTY LLC              3995.0           4  0.92   0.93  0.84  1.00   
1 Team Realtors             1500000.0          13  1.01   1.00  0.97  1.08   
1 Vision Real Estate        1412500.0          11  0.98   1.00  0.87  1.06   
1% LISTING FEE              2390000.0          36  0.99   0.99  0.84  1.18   
1% Listing Broker           1912500.0          64  1.05   1.05  0.83  1.33   
1-2-3 Homekeys               824000.0           1  0.99   0.99  0.99  0.99   
10-8 Real Estate            2500000.0           4  1.05   1.03  0.96  1.17   
10-Point-o Inc.             1900000.0           2  0.95   0.95  0.87  1.02   
100 Doors Realty, Inc.       962700.0          10  0.88   0.97  0.00  1.07   
1000 Realty Corporation     2800000.0          43  0.98   1.00  0.65  1.14   
101 Realty Services         5558888.0           3  1.00   1.00  0.89  1.11   
101 Realty, Inc.            2150000.0           3  0.99   0.96  0.91  1.08   
10X Realty Mortgage         1620000.0          29  0.99   1.00  0.84  1.10   

                            ... listing_to_contract_days                      \
                            ...                    count   mean median   min   
ListOfficeName              ...                                                
#1 FLAT FEE-LIBERTY REALTY  ...                        2  20.50   20.5  17.0   
#1 Home Sales®              ...                        2  54.50   54.5  13.0   
#1 Home SalesÂ®             ...                        1  78.00   78.0  78.0   
#1 Realty Homes Inc.        ...                  


Sold Segment Summary by BuyerOfficeName


ClosePrice                                    \
                            count        mean     median        min   
BuyerOfficeName                                                       
#1 Home SalesÂ®                 1   545000.00   545000.0   545000.0   
#1 Realty Homes Inc.            7     2514.29     2500.0     1700.0   
& Company Real Estate           2     3575.00     3575.0     3500.0   
*                               6   548750.00   634500.0     3500.0   
, Inc                           1  2225000.00  2225000.0  2225000.0   
01328848                        1   675000.00   675000.0   675000.0   
01718402                        1   515000.00   515000.0   515000.0   
01719850                        1   800000.00   800000.0   800000.0   
1                               1   895000.00   895000.0   895000.0   
1 Percent Lists SoCal          30   732364.00   785412.5     2850.0   
1 Realty                       16   263743.00   162500.0     2400.0   
1 STOP REALTY LLC               5   232628.00     3600.0     2295.0   
1 Team Realtors                18   840188.89   800000.0   205000.0   
1 Vision Real Estate           25   654839.60   645000.0     3000.0   
1% LISTING FEE                 20   670610.00   666500.0     2000.0   
1% Listing Broker              21   936904.76  1025000.0     4500.0   
1% Listing Fee                  1   780000.00   780000.0   780000.0   
1-2-3 Homekeys                  1   773990.00   773990.0   773990.0   
1-888-CASH-OFFER, INC.          2  1254500.00  1254500.0   900000.0   
10-8 Real Estate                1  1920000.00  1920000.0  1920000.0   

                                  price_ratio                           ...  \
                              max       count  mean median   min   max  ...   
BuyerOfficeName                                                         ...   
#1 Home SalesÂ®          545000.0           1  0.84   0.84  0.84  0.84  ...   
#1 Realty Homes Inc.       3500.0           7  0.98   1.00  0.91  1.00  ...   
& Company Real Estate      3650.0           2  1.00   1.00  1.00  1.00  ...   
*                        865000.0           6  0.94   0.98  0.72  1.12  ...   
, Inc                   2225000.0           1  1.06   1.06  1.06  1.06  ...   
01328848                 675000.0           1  0.93   0.93  0.93  0.93  ...   
01718402                 515000.0           1  1.05   1.05  1.05  1.05  ...   
01719850                 800000.0           1  1.02   1.02  1.02  1.02  ...   
1                        895000.0           1  0.97   0.97  0.97  0.97  ...   
1 Percent Lists SoCal   2525000.0          30  0.99   1.00  0.78  1.08  ...   
1 Realty                 868000.0          16  0.98   0.97  0.91  1.09  ...   
1 STOP REALTY LLC       1150000.0           5  0.92   0.92  0.84  1.00  ...   
1 Team Realtors         1750000.0          17  0.96   0.99  0.59  1.07  ...   
1 Vision Real Estate    1380000.0          25  1.01   1.00  0.82  1.15  ...   
1% LISTING FEE          1550000.0          20  1.00   1.00  0.88  1.18  ...   
1% Listing Broker       1840000.0          21  1.06   1.06  0.89  1.23  ...   
1% Listing Fee           780000.0           1  0.92   0.92  0.92  0.92  ...   
1-2-3 Homekeys           773990.0           1  1.01   1.01  1.01  1.01  ...   
1-888-CASH-OFFER, INC.  1609000.0           2  0.91   0.91  0.87  0.95  ...   
10-8 Real Estate        1920000.0           1  0.97   0.97  0.97  0.97  ...   

                       listing_to_contract_days                               \
                                          count    mean median    min    max   
BuyerOfficeName                                                                
#1 Home SalesÂ®                               1   78.00   78.0   78.0   78.0   
#1 Realty Homes Inc.                          1   64.00   64.0   64.0   64.0   
& Company Real Estate                         2   10.00   10.0    9.0   11.0   
*                                             6   77.50   76.5   17.0  137.0   
, Inc      

In [15]:
# Data quality check summary

print("\n==============================")
print("Data Quality Flags - Sold Data")
print("==============================")

flag_cols = [
    "extreme_price_ratio_flag",
    "negative_contract_to_close_flag",
    "negative_listing_to_contract_flag"
]

for col in flag_cols:
    if col in sold_fe.columns:
        print(f"{col}: {sold_fe[col].sum()}")


Data Quality Flags - Sold Data
extreme_price_ratio_flag: 1304
negative_contract_to_close_flag: 492
negative_listing_to_contract_flag: 433


In [16]:
# Save final feature-engineered datasets

sold_fe.to_csv("sold_feature_engineered.csv", index=False)
listings_fe.to_csv("listings_feature_engineered.csv", index=False)

# Save segmented summary tables
if property_type_summary is not None:
    property_type_summary.to_csv("property_type_summary.csv")

if county_summary is not None:
    county_summary.to_csv("county_summary.csv")

print("Feature engineering complete. Files saved successfully.")

Feature engineering complete. Files saved successfully.
